# New Preprocessing - fix overlapping subjects

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

In [5]:
# File paths
calsnic1_path = "/Users/gracieshao/Desktop/ALS T1s/LIFEx_results/CALSNIC1_global_texture_binsize32/TEXTURE_ROI_Results_CLASNIC1.csv"

calsnic2_path = "/Users/gracieshao/Desktop/ALS T1s/LIFEx_results/CALSNIC2_global_texture_binsize32/TEXTURE_ROI_Results_CALSNIC2.csv"

# Read CSVs
df1 = pd.read_csv(calsnic1_path)
df2 = pd.read_csv(calsnic2_path)

# Optional: add dataset labels
df1["Dataset"] = "CALSNIC1"
df2["Dataset"] = "CALSNIC2"

# Combine
combined_df = pd.concat([df1, df2], ignore_index=True)

# # Save combined file
output_path = "/Users/gracieshao/Desktop/ALS T1s/LIFEx_results/TEXTURE_ROI_Results_COMBINED.csv"

combined_df.to_csv(output_path, index=False)

print("Combined shape:", combined_df.shape)
print("Saved to:", output_path)

/var/folders/sr/cyglh3yj7gv2wll3d8p91sz00000gn/T/ipykernel_96178/1736970093.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df1["Dataset"] = "CALSNIC1"
/var/folders/sr/cyglh3yj7gv2wll3d8p91sz00000gn/T/ipykernel_96178/1736970093.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df2["Dataset"] = "CALSNIC2"


Combined shape: (117576, 352)
Saved to: /Users/gracieshao/Desktop/ALS T1s/LIFEx_results/TEXTURE_ROI_Results_COMBINED.csv


In [6]:
print(df1.columns.equals(df2.columns))

True


In [7]:
#drop non-used features
cols_to_drop = [
    #processed time
    "INFO_ActualFrameDuration", 
    #duplicate file name
    "INFO_Filter",
    "INFO_Series",
    #file path
    "INFO_SeriesPath",
    #all 9999/1/1
    "INFO_SeriesDate",
    #processed date
    "INFO_ProcessDateOfTexture",
    #all 0
    "INFO_PatientID",
    #software info
    "INFORMATION",

    "PARAMETERS",
    "PARAMS1_ZYXspacing[mm]",
    "PARAMS2_SPATIALFILTER_AppliedFilterType",

    #all non-used
    "PARAMS2_SPATIALFILTER_LaplacianOfGaussianFilter_Sigma_X[mm]",
    "PARAMS2_SPATIALFILTER_LaplacianOfGaussianFilter_Sigma_Y[mm]",
    "PARAMS2_SPATIALFILTER_LaplacianOfGaussianFilter_Sigma_Z[mm]",
    "PARAMS2_SPATIALFILTER_LaplacianOfGaussianFilter_CalculationDimension",
    "PARAMS2_SPATIALFILTER_LaplacianOfGaussianFilter_BoundingBox",
    "PARAMS2_SPATIALFILTER_LaplacianOfGaussianFilter_PaddingMethod",
    "PARAMS2_SPATIALFILTER_GaussianFilter_Sigma_X[mm]",
    "PARAMS2_SPATIALFILTER_GaussianFilter_Sigma_Y[mm]",
    "PARAMS2_SPATIALFILTER_GaussianFilter_Sigma_Z[mm]",
    "PARAMS2_SPATIALFILTER_GaussianFilter_CalculationDimension",
    "PARAMS2_SPATIALFILTER_GaussianFilter_BoundingBox",
    "PARAMS2_SPATIALFILTER_GaussianFilter_PaddingMethod",
    "PARAMS2_SPATIALFILTER_LawsFilter_Kernel_X[vx]",
    "PARAMS2_SPATIALFILTER_LawsFilter_Kernel_Y[vx]",
    "PARAMS2_SPATIALFILTER_LawsFilter_Kernel_Z[vx]",
    "PARAMS2_SPATIALFILTER_LawsFilter_BoundingBox",
    "PARAMS2_SPATIALFILTER_LawsFilter_PaddingMethod",
    "PARAMS2_SPATIALFILTER_MeanFilter_KernelSize[vx]",
    "PARAMS2_SPATIALFILTER_MeanFilter_CalculationDimension",
    "PARAMS2_SPATIALFILTER_MeanFilter_BoundingBox",
    "PARAMS2_SPATIALFILTER_MeanFilter_PaddingMethod",
    "PARAMS2_SPATIALFILTER_WaveletFilter_TransformAlong",
    "PARAMS2_SPATIALFILTER_WaveletFilter_BoundingBox",
    "PARAMS2_SPATIALFILTER_WaveletFilter_PaddingMethod",
    "PARAMS2_SPATIALFILTER_WaveletFilter_Family",
    "PARAMS2_SPATIALFILTER_WaveletFilter_Order",
    "PARAMS2_SPATIALFILTER_WaveletFilter_Level",

    
    "PARAMS3_TEXTURE_BinSize", #binsize, all =32
    "PARAMS4_TEXTURE_IntensityPeakDiscretizedVolumeSought(IBSI:No)[mL]",

    #not features
    "PARAMS4_TEXTURE_IntensityRescaling",
    "PARAMS4_TEXTURE_BoundsRangeOfValueAfterDiscretisation()",
    
    "PARAMS5_TEXTURE_DimensionProcessing", #all =3d
    "PARAMS5_TEXTURE_isKeepTheLargestCluster", #all =FALSE
    "PARAMS5_TEXTURE_GLCM_DistanceOfNeighbours", #all =1
    "CHECK_TEXTURE_SliceTimePosition[#]", #all =0
    "CHECK_TEXTURE_Location(2DProcessingOnly)[vx]",
    "CHECK_TEXTURE_Cluster(s)ToSmall", #empty
    "FEATURE RESULTS", #empty
    "MORPHOLOGICAL_PeakIntensityCoor-RoiCentroidCoor-Dist(IBSI:No)[cm]", #all NaN
    
    #a lot of missing values, 0
    "MORPHOLOGICAL_MaxIntensityCoor-PerimeterCoor-3DSmallestDist(IBSI:No)[mm]",
    "MORPHOLOGICAL_RadiusSphereNorm-MaxIntensityCoor-PerimeterCoor-3DSmallestDist(IBSI:No)[]",
    "MORPHOLOGICAL_RadiusRoiNorm-MaxIntensityCoor-PerimeterCoor-3DSmallestDist(IBSI:No)[]",
    "MORPHOLOGICAL_MaxIntensityCoor-PerimeterCoor-2DAxialSmallestDist(IBSI:No)[mm]",
    "MORPHOLOGICAL_RadiusSphereNorm-MaxIntensityCoor-PerimeterCoor-2DAxialSmallestDist(IBSI:No)[]",
    "MORPHOLOGICAL_RadiusRoiNorm-MaxIntensityCoor-PerimeterCoor-2DAxialSmallestDist(IBSI:No)[]",
    "MORPHOLOGICAL_MaxIntensityCoor-PerimeterCoor-2DCoronalSmallestDist(IBSI:No)[mm]",
    "MORPHOLOGICAL_RadiusSphereNorm-MaxIntensityCoor-PerimeterCoor-2DCoronalSmallestDist(IBSI:No)[]",
    "MORPHOLOGICAL_RadiusRoiNorm-MaxIntensityCoor-PerimeterCoor-2DCoronalSmallestDist(IBSI:No)[]",
    "MORPHOLOGICAL_MaxIntensityCoor-PerimeterCoor-2DSagittalSmallestDist(IBSI:No)[mm]",
    "MORPHOLOGICAL_RadiusSphereNorm-MaxIntensityCoor-PerimeterCoor-2DSagittalSmallestDist(IBSI:No)[]",
    "MORPHOLOGICAL_RadiusRoiNorm-MaxIntensityCoor-PerimeterCoor-2DSagittalSmallestDist(IBSI:No)[]",

    #all NaNs
    "MORPHOLOGICAL_RadiusSphereNorm-PeakIntensityCoor-RoiCentroidCoor-Dist(IBSI:No)[]",
    "MORPHOLOGICAL_RadiusRoiNorm-PeakIntensityCoor-RoiCentroidCoor-Dist( (IBSI:No)[]",
    "MORPHOLOGICAL_PeakIntensityCoor-PerimeterCoor-3DSmallestDist(IBSI:No)[mm]",
    "MORPHOLOGICAL_RadiusSphereNorm-PeakIntensityCoor-PerimeterCoor-3DSmallestDist(IBSI:No)[]",
    "MORPHOLOGICAL_RadiusRoiNorm-PeakIntensityCoor-PerimeterCoor-3DSmallestDist(IBSI:No)[]",
    "MORPHOLOGICAL_PeakIntensityCoor-PerimeterCoor-2DAxialSmallestDist(IBSI:No)[mm]",
    "MORPHOLOGICAL_RadiusSphereNorm-PeakIntensityCoor-PerimeterCoor-2DAxialSmallestDist(IBSI:No)[]",
    "MORPHOLOGICAL_RadiusRoiNorm-PeakIntensityCoor-PerimeterCoor-2DAxialSmallestDist(IBSI:No)[]",
    "MORPHOLOGICAL_PeakIntensityCoor-PerimeterCoor-2DCoronalSmallestDist(IBSI:No)[mm]",
    "MORPHOLOGICAL_RadiusSphereNorm-PeakIntensityCoor-PerimeterCoor-2DCoronalSmallestDist(IBSI:No)[]",
    "MORPHOLOGICAL_RadiusRoiNorm-PeakIntensityCoor-PerimeterCoor-2DCoronalSmallestDist(IBSI:No)[]",
    "MORPHOLOGICAL_PeakIntensityCoor-PerimeterCoor-2DSagittalSmallestDist(IBSI:No)[mm]",
    "MORPHOLOGICAL_RadiusSphereNorm-PeakIntensityCoor-PerimeterCoor-2DSagittalSmallestDist(IBSI:No)[]",
    "MORPHOLOGICAL_RadiusRoiNorm-PeakIntensityCoor-PerimeterCoor-2DSagittalSmallestDist(IBSI:No)[]",

    #all NaNs
    "INTENSITY-BASED_TotalCalciumScore[OnlyOnCT](IBSI:No)[]",
    "LOCAL_INTENSITY_BASED_GlobalIntensityPeak(IBSI:0F91)[Intensity]",
    "LOCAL_INTENSITY_BASED_LocalIntensityPeak(IBSI:VJGA)[Intensity]",


    "INTENSITY-BASED-RIM_RIM-IntensityMin(IBSI:No)[Intensity]", #measures boundary artifacts, not tissue properties

    "INTENSITY-BASED-RIM_RIM-CountingVoxels_Min(IBSI:No)[vx]", #all zeros

    #all NaNs
    "LOCAL_INTENSITY_HISTOGRAM_GlobalIntensityPeak(IBSI:No)[Intensity]",
    "LOCAL_INTENSITY_HISTOGRAM_LocalIntensityPeak(IBSI:No)[Intensity]",

    #a lot of NaNs
    "INTENSITY-HISTOGRAM_MaximumHistogramGradient(IBSI:12CE)[Intensity]",
    "INTENSITY-HISTOGRAM_MaximumHistogramGradientGreyLevel(IBSI:8E6O)[Intensity]",
    "INTENSITY-HISTOGRAM_MinimumHistogramGradient(IBSI:VQB3)[Intensity]",
    "INTENSITY-HISTOGRAM_MinimumHistogramGradientGreyLevel(IBSI:RHQZ)[Intensity]",

    "INTENSITY-HISTOGRAM-RIM_RIM-IntensityMin(IBSI:No)[Intensity]", #hard to implement in PCA, RIM
    "INTENSITY-HISTOGRAM-RIM_RIM-IntensityMin_Min(IBSI:No)[Intensity]", #all 1

    "INTENSITY-HISTOGRAM-RIM_RIM-IntensityMin-GradientMin(IBSI:No)[Intensity]" #many zeros

    "INTENSITY-HISTOGRAM-RIM_RIM-CountingVoxels(IBSI:No)[vx]", #RIM
    "INTENSITY-HISTOGRAM-RIM_RIM-CountingVoxels_Min(IBSI:No)[vx]", #all zeros

    "INTENSITY-HISTOGRAM-RIM_RIM-ApproximateVolume(IBSI:No)[mL]", #RIM

    "INTENSITY-BASED_IntensityBasedQuartileCoefficientOfDispersion(IBSI:9S40)[]" #1250 missing values
]

combined_df = combined_df.drop(columns=cols_to_drop, errors='ignore')

In [8]:
combined_df.head(10)

,INFO_PatientName,INFO_NameOfRoi,PARAMS3_TEXTURE_NbGreyLevels,MORPHOLOGICAL_Volume(IBSI:RNU0)[cm3],MORPHOLOGICAL_ApproximateVolume(IBSI:YEKZ)[cm3],MORPHOLOGICAL_voxelsCounting(IBSI:No)[#vx],MORPHOLOGICAL_SurfaceArea(IBSI:C0JK)[cm2],MORPHOLOGICAL_SurfaceToVolumeRatio(IBSI:2PR5)[cm],MORPHOLOGICAL_Compacity(IBSI:No)[],MORPHOLOGICAL_Compactness1(IBSI:SKGS)[],...,GLSZM_LargeZoneHighGreyLevelEmphasis(IBSI:J17V),GLSZM_GreyLevelNonUniformity(IBSI:JNSA),GLSZM_NormalisedGreyLevelNonUniformity(IBSI:Y1RO),GLSZM_ZoneSizeNonUniformity(IBSI:4JP3),GLSZM_NormalisedZoneSizeNonUniformity(IBSI:VB3A),GLSZM_ZonePercentage(IBSI:P30P),GLSZM_GreyLevelVariance(IBSI:BYLV),GLSZM_ZoneSizeVariance(IBSI:3NSA),GLSZM_ZoneSizeEntropy(IBSI:GU8N),Dataset
0,CALSNIC1_CAL_C001_T1w10_V1_n4_MNI_gmwm_hm,JHU-WhiteMatter-labels-1mm_#47,28.476379,0.579542,0.596,596,5.652878,9.754049,23.191014,0.024328,...,6.676728e+03,13.092166,0.060333,80.032258,0.368812,0.364706,23.910043,15.739854,5.824657,CALSNIC1
1,CALSNIC1_CAL_C001_T1w10_V1_n4_MNI_gmwm_hm,harvardoxford-cortical_label_all_#26,239.312546,11.914208,11.954,11954,37.593606,3.155359,19.346650,0.029162,...,1.338021e+05,77.618104,0.012706,3541.430185,0.579707,0.511085,515.584308,1303.819529,7.654397,CALSNIC1
2,CALSNIC1_CAL_C001_T1w10_V1_n4_MNI_gmwm_hm,harvardoxford-cortical_label_all_#32,237.467270,9.872208,9.934,9934,37.966393,3.845785,23.696527,0.023809,...,1.686559e+05,77.971352,0.012987,3529.239507,0.587815,0.604450,529.999397,126.652877,7.581374,CALSNIC1
3,CALSNIC1_CAL_C001_T1w10_V1_n4_MNI_gmwm_hm,harvardoxford-cortical_label_all_#11,242.013962,6.937792,6.952,6952,30.954183,4.461677,24.823200,0.022728,...,1.384130e+05,44.404188,0.011769,2024.441293,0.536560,0.542800,620.014113,148.099435,7.812382,CALSNIC1
4,CALSNIC1_CAL_C001_T1w10_V1_n4_MNI_gmwm_hm,harvardoxford-cortical_label_all_#37,241.185440,4.852292,4.901,4901,22.883055,4.715927,22.559222,0.025009,...,2.038022e+05,37.365325,0.014460,1248.037926,0.482987,0.527347,503.347784,16.666488,7.786208,CALSNIC1
5,CALSNIC1_CAL_C001_T1w10_V1_n4_MNI_gmwm_hm,JHU-WhiteMatter-labels-1mm_#46,47.311356,0.364708,0.376,376,3.332745,9.138110,16.682357,0.033820,...,3.331054e+03,5.763441,0.030986,80.483871,0.432709,0.496000,91.774540,4.091138,6.279653,CALSNIC1
6,CALSNIC1_CAL_C001_T1w10_V1_n4_MNI_gmwm_hm,JHU-WhiteMatter-labels-1mm_#28,246.058502,3.680250,3.714,3714,22.205016,6.033562,28.431471,0.019844,...,1.393560e+06,39.048048,0.029315,528.400901,0.396697,0.358740,316.351141,32.029184,7.348330,CALSNIC1
7,CALSNIC1_CAL_C001_T1w10_V1_n4_MNI_gmwm_hm,JHU-WhiteMatter-labels-1mm_#37,60.992294,1.207292,1.236,1236,10.233516,8.476424,27.115966,0.020807,...,5.747408e+03,15.082079,0.020632,368.357045,0.503908,0.591903,181.880777,2.424773,6.916403,CALSNIC1
8,CALSNIC1_CAL_C001_T1w10_V1_n4_MNI_gmwm_hm,harvardoxford-subcortical_label_all_#8,252.637604,38.558208,38.610,38610,77.164378,2.001244,17.579578,0.032093,...,1.331731e+06,148.000145,0.010759,7545.303431,0.548510,0.356290,770.073944,6746.064192,8.147197,CALSNIC1
9,CALSNIC1_CAL_C001_T1w10_V1_n4_MNI_gmwm_hm,JHU-WhiteMatter-labels-1mm_#40,249.177628,1.100208,1.125,1125,9.159663,8.325390,25.196738,0.022391,...,1.716909e+05,8.487730,0.013018,409.239264,0.627668,0.580071,736.196100,36.316421,7.253870,CALSNIC1


# Remove features 

In [10]:
# Separate metadata from imaging features
meta_cols = [c for c in [
    "INFO_PatientName", "INFO_NameOfRoi"
] if c in combined_df.columns]

meta_df    = combined_df[meta_cols].copy()
feature_df = combined_df.drop(columns=meta_cols, errors="ignore").select_dtypes(include=[np.number]).copy()

print("Metadata shape:", meta_df.shape)
print("Feature shape before screening:", feature_df.shape)

Metadata shape: (117576, 2)
Feature shape before screening: (117576, 243)


In [11]:
after_drop = combined_df.drop(columns=meta_cols, errors="ignore")

non_numeric = after_drop.select_dtypes(exclude=[np.number])

print(non_numeric.columns.tolist())
print(len(non_numeric.columns))

['MORPHOLOGICAL_MaxIntensityCoor(IBSI:No)[vx]', 'MORPHOLOGICAL_Centroid(IBSI:No)[vx]', 'MORPHOLOGICAL_WeightedCenterOfMass(IBSI:No)[vx]', 'INTENSITY-BASED-RIM_RIM-IntensityMean(IBSI:No)[Intensity]', 'INTENSITY-BASED-RIM_RIM-IntensityStd(IBSI:No)[Intensity]', 'INTENSITY-BASED-RIM_RIM-IntensityMax(IBSI:No)[Intensity]', 'INTENSITY-BASED-RIM_RIM-CountingVoxels(IBSI:No)[vx]', 'INTENSITY-BASED-RIM_RIM-ApproximateVolume(IBSI:No)[mL]', 'INTENSITY-BASED-RIM_RIM-IntensitySum(IBSI:No)[Intensity]', 'INTENSITY-HISTOGRAM-RIM_RIM-IntensityMean(IBSI:No)[Intensity]', 'INTENSITY-HISTOGRAM-RIM_RIM-IntensityStd(IBSI:No)[Intensity]', 'INTENSITY-HISTOGRAM-RIM_RIM-IntensityMax(IBSI:No)[Intensity]', 'INTENSITY-HISTOGRAM-RIM_RIM-CountingVoxels(IBSI:No)[vx]', 'INTENSITY-HISTOGRAM-RIM_RIM-IntensitySum(IBSI:No)[Intensity]', 'Dataset']
15


In [13]:
# ── Step 1: Remove zero/near-zero variance on ALL subjects (own cohort) ──────
variance = feature_df.var(axis=0)
nzv_cols = variance[variance <= 1e-5].index
print("Near-zero variance features to drop:", len(nzv_cols))

filtered_df = feature_df.drop(columns=nzv_cols)
print("Features after near-zero variance filtering:", filtered_df.shape[1])

Near-zero variance features to drop: 2
Features after near-zero variance filtering: 241


In [14]:
# ── Step 2: Remove highly correlated on ALL subjects (own cohort) ─────────────
# Keeps the first feature in each correlated cluster, drops the rest
corr_matrix = filtered_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
corr_drop_cols = [col for col in upper.columns if any(upper[col] > 0.95)]
print("Highly correlated features to drop:", len(corr_drop_cols))

filtered_df = filtered_df.drop(columns=corr_drop_cols)
print("Features after correlation filtering:", filtered_df.shape[1])

Highly correlated features to drop: 141
Features after correlation filtering: 100


In [15]:
filtered_df.isna().sum().sum()

np.int64(52)

In [11]:
# ── Step 3: Z-score using control mean and SD ─────────────────────────────────
# In INFO_PatientName:
# CALSNIC1_CAL_C001_... = Control
# CALSNIC1_VAN_P012_... = Patient

subject_code = meta_df["INFO_PatientName"].str.extract(r"CALSNIC[12]_[A-Z]+_([CP]\d+)")[0]

control_mask = subject_code.str.startswith("C", na=False)
patient_mask = subject_code.str.startswith("P", na=False)

control_mean     = filtered_df.loc[control_mask].mean(axis=0)
control_std      = filtered_df.loc[control_mask].std(axis=0, ddof=1)
control_std_safe = control_std.replace(0, np.nan)

z_feature_df = (filtered_df - control_mean) / control_std_safe

print(f"Controls used as reference: {control_mask.sum()}")
print(f"Patients z-scored:          {patient_mask.sum()}")
print(f"Unknown rows:               {(~control_mask & ~patient_mask).sum()}")
print(f"NaNs after z-scoring:       {z_feature_df.isna().sum().sum()}")
print(f"Final retained features:    {z_feature_df.shape[1]}")

Controls used as reference: 55925
Patients z-scored:          61651
Unknown rows:               0
NaNs after z-scoring:       52
Final retained features:    100


In [12]:
# Per-feature: how many rows are NaN in each column?
nan_per_feature = z_feature_df.isna().sum()
nan_features = nan_per_feature[nan_per_feature > 0]
print(f"Features with any NaN: {len(nan_features)}")
print(nan_features)

print()

# Per-subject: how many features are NaN for each row?
nan_per_row = z_feature_df.isna().sum(axis=1)
nan_rows = nan_per_row[nan_per_row > 0]
print(f"Rows with any NaN: {len(nan_rows)}")
print("NaN count distribution across affected rows:")
print(nan_rows.value_counts().sort_index())

Features with any NaN: 26
GLCM_DifferenceAverage(IBSI:TF7R)                    2
GLCM_NormalisedInverseDifference(IBSI:NDRX)          2
GLCM_NormalisedInverseDifferenceMoment(IBSI:1QCO)    2
GLCM_InverseVariance(IBSI:E8JP)                      2
GLCM_Correlation(IBSI:NI2N)                          2
GLCM_ClusterShade(IBSI:7NFM)                         2
GLCM_ClusterProminence(IBSI:AE86)                    2
GLRLM_ShortRunLowGreyLevelEmphasis(IBSI:HTZT)        2
GLRLM_GreyLevelVariance(IBSI:8CE5)                   2
GLRLM_RunEntropy(IBSI:HJ90)                          2
NGTDM_Coarseness(IBSI:QCDE)                          2
NGTDM_Contrast(IBSI:65HE)                            2
NGTDM_Busyness(IBSI:NQ30)                            2
NGTDM_Complexity(IBSI:HDEZ)                          2
NGTDM_Strength(IBSI:1X9X)                            2
GLSZM_SmallZoneEmphasis(IBSI:5QRC)                   2
GLSZM_LargeZoneEmphasis(IBSI:48P8)                   2
GLSZM_LowGrayLevelZoneEmphasis(IBSI:XMS

In [13]:
# ── Identify and drop the 2 NaN rows ──────────────────────────────────────────
nan_row_idx = z_feature_df[z_feature_df.isna().any(axis=1)].index
print("Affected rows:")
print(meta_df.loc[nan_row_idx])

# Drop from both feature and metadata DataFrames
z_feature_df = z_feature_df.drop(index=nan_row_idx)
meta_df      = meta_df.drop(index=nan_row_idx)

print(f"\nRows dropped: {len(nan_row_idx)}")
print(f"Remaining rows: {z_feature_df.shape[0]}")

Affected rows:
                                INFO_PatientName  \
67136  CALSNIC2_QUE_C024_T1w10_V2_n4_MNI_gmwm_hm   
72704  CALSNIC2_TOR_C038_T1w10_V2_n4_MNI_gmwm_hm   

                       INFO_NameOfRoi  
67136  JHU-WhiteMatter-labels-1mm_#46  
72704  JHU-WhiteMatter-labels-1mm_#11  

Rows dropped: 2
Remaining rows: 117574


# Save out

In [19]:
final_df = pd.concat(
    [meta_df.reset_index(drop=True), z_feature_df.reset_index(drop=True)],
    axis=1
)

# Remove CAL P003 from CALSNIC1 and MIA P024 from CALSNIC2

exclude_mask = (
    final_df['INFO_PatientName'].str.contains('CALSNIC1_CAL_P003', na=False) |
    final_df['INFO_PatientName'].str.contains('CALSNIC2_MIA_P024', na=False) |
    final_df['INFO_PatientName'].str.contains('CALSNIC2_EDM_P103', na=False)
)

# Keep all other rows
final_df = final_df[~exclude_mask].copy()

# Optional sanity check
print(
    final_df['INFO_PatientName']
    .loc[
        final_df['INFO_PatientName'].str.contains(
            'CALSNIC1_CAL_P003|CALSNIC2_MIA_P024|CALSNIC2_EDM_P103',
            na=False
        )
    ]
)

print("Final dataframe shape:", final_df.shape)

final_df.to_csv("/Users/gracieshao/Desktop/ALS T1s/data/preprocessed/final_cleaned_zscored_features_COMBINED.csv", index=False)

Series([], Name: INFO_PatientName, dtype: str)
Final dataframe shape: (117226, 102)


In [15]:
# ── Create subject-level identifiers ──────────────────────────────────────────
# Example:
# CALSNIC1_CAL_C001_T1w10_V1_n4_MNI_gmwm_hm
# → Study = CALSNIC1
# → SubjectID = C001 / P001

final_df['Study'] = final_df['INFO_PatientName'].str.extract(r'^(CALSNIC[12])')
final_df['SubjectCode'] = final_df['INFO_PatientName'].str.extract(
    r'CALSNIC[12]_[A-Z]+_([CP]\d+)'
)

# Patient vs Control
final_df['Group'] = np.where(
    final_df['SubjectCode'].str.startswith('C'),
    'Control',
    'Patient'
)

# Unique subject table (collapses V1/V2/V3 into one subject)
subject_df = final_df[['Study', 'SubjectCode', 'Group']].drop_duplicates()

# ── Count totals ──────────────────────────────────────────────────────────────
summary = (
    subject_df
    .groupby(['Study', 'Group'])
    .size()
    .unstack(fill_value=0)
)

summary['Total'] = summary.sum(axis=1)

print(summary)

# Optional overall totals
print("\nOverall totals:")
print(subject_df['Group'].value_counts())
print("\nTotal unique subjects:", len(subject_df))

Group     Control  Patient  Total
Study                            
CALSNIC1       21       24     45
CALSNIC2       84       86    170

Overall totals:
Group
Patient    110
Control    105
Name: count, dtype: int64

Total unique subjects: 215


In [17]:
import pandas as pd

# ── Parse all info from INFO_PatientName ──────────────────────────────────────
# Format: CALSNIC1_CAL_C001_T1w10_V1_n4_MNI_gmwm_hm
#          [0]     [1]  [2]  ...  [4] ...

parts = final_df['INFO_PatientName'].str.split('_')

final_df['Study']      = parts.str[0]                          # CALSNIC1 / CALSNIC2
final_df['SiteCode']   = parts.str[1]                          # CAL, EDM, etc.
final_df['SubjectID']  = parts.str[2]                          # C001, P001, etc.
final_df['Visit']      = parts.str.get(4)                      # V1, V2, V3
final_df['Group']      = final_df['SubjectID'].str[0].map(
                              {'P': 'Patient', 'C': 'Control'})

# ── Unique subjects (one row per person per cohort) ───────────────────────────
unique_subj = (
    final_df
    .drop_duplicates(subset=['Study', 'SiteCode', 'SubjectID'])
    [['Study', 'SiteCode', 'SubjectID', 'Group']]
    .copy()
)

# ── Summary table ─────────────────────────────────────────────────────────────
summary = (
    unique_subj
    .groupby(['Study', 'Group'])
    .size()
    .unstack(fill_value=0)
    .assign(Total=lambda df: df.sum(axis=1))
)
summary.loc['ALL'] = summary.sum()

print("=== Unique subjects in final_df ===\n")
print(summary.to_string())

# ── Cross-cohort overlap (same person in both CALSNIC1 and CALSNIC2) ──────────
c1_keys = set(
    unique_subj[unique_subj['Study'] == 'CALSNIC1']['SiteCode'] +
    unique_subj[unique_subj['Study'] == 'CALSNIC1']['SubjectID'].str[1:]
)
c2_keys = set(
    unique_subj[unique_subj['Study'] == 'CALSNIC2']['SiteCode'] +
    unique_subj[unique_subj['Study'] == 'CALSNIC2']['SubjectID'].str[1:]
)
overlap = c1_keys & c2_keys

print(f"\nSubjects in both cohorts:         {len(overlap)}")
print(f"True unique individuals (no dup): {len(c1_keys) + len(c2_keys) - len(overlap)}")

# ── Visit distribution ────────────────────────────────────────────────────────
print("\n=== Visits available per cohort × group ===")
visit_summary = (
    final_df
    .drop_duplicates(subset=['Study', 'SiteCode', 'SubjectID', 'Visit'])
    .groupby(['Study', 'Group', 'Visit'])
    .size()
    .unstack(fill_value=0)
)
print(visit_summary.to_string())

=== Unique subjects in final_df ===

Group     Control  Patient  Total
Study                            
CALSNIC1       63       82    145
CALSNIC2      143      171    314
ALL           206      253    459

Subjects in both cohorts:         69
True unique individuals (no dup): 333

=== Visits available per cohort × group ===
Visit              V1   V2  V3
Study    Group                
CALSNIC1 Control   63   48  36
         Patient   82   60  43
CALSNIC2 Control  141  107  83
         Patient  170  103  67


In [18]:
import pandas as pd

# ── Parse INFO_PatientName ────────────────────────────────────────────────────
parts = final_df['INFO_PatientName'].str.split('_')

final_df['Study']     = parts.str[0]          # CALSNIC1 / CALSNIC2
final_df['SiteCode']  = parts.str[1]          # CAL, EDM, etc.
final_df['SubjectID'] = parts.str[2]          # C001, P001, etc.
final_df['Visit']     = parts.str.get(4)      # V1, V2, V3
final_df['Group']     = final_df['SubjectID'].str[0].map(
                            {'P': 'Patient', 'C': 'Control'})

# ── Unique subject = Study + SiteCode + SubjectID (cohort is part of identity)
unique_subj = (
    final_df
    .drop_duplicates(subset=['Study', 'SiteCode', 'SubjectID'])
    [['Study', 'SiteCode', 'SubjectID', 'Group']]
    .copy()
)

# ── Summary table ─────────────────────────────────────────────────────────────
summary = (
    unique_subj
    .groupby(['Study', 'Group'])
    .size()
    .unstack(fill_value=0)
    .assign(Total=lambda df: df.sum(axis=1))
)
summary.loc['ALL'] = summary.sum()

print("=== Unique subjects in final_df ===")
print("(Study + SiteCode + SubjectID as unique identifier)\n")
print(summary.to_string())

# ── Visit distribution ────────────────────────────────────────────────────────
print("\n=== Visits available per cohort × group ===")
visit_summary = (
    final_df
    .drop_duplicates(subset=['Study', 'SiteCode', 'SubjectID', 'Visit'])
    .groupby(['Study', 'Group', 'Visit'])
    .size()
    .unstack(fill_value=0)
)
print(visit_summary.to_string())

=== Unique subjects in final_df ===
(Study + SiteCode + SubjectID as unique identifier)

Group     Control  Patient  Total
Study                            
CALSNIC1       63       82    145
CALSNIC2      143      171    314
ALL           206      253    459

=== Visits available per cohort × group ===
Visit              V1   V2  V3
Study    Group                
CALSNIC1 Control   63   48  36
         Patient   82   60  43
CALSNIC2 Control  141  107  83
         Patient  170  103  67
